# Retrieval-Augmented Generation, from scratch

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

A language model's knowledge is frozen at training time, and it cannot tell you where an answer came from. **Retrieval-Augmented Generation** fixes both: before answering, the system looks up relevant passages in documents *you* control, and answers from those.

> **Analogy.** RAG is to an LLM what an open-book exam is to a student. They are not expected to have memorised everything; they are allowed to look it up — and to show you which page.

This notebook builds one from nothing. No RAG framework, no vector database: a HuggingFace pipeline, a sentence-embedding model, NumPy, and about a hundred lines of ordinary Python. A "vector store" will turn out to be a matrix. A "retriever" will turn out to be a sort. Seeing that once is what lets you debug the frameworks later, and decide when you need one.

| Section | |
|---|---|
| 1–2 | Setup: a generator and an embedder |
| 3 | The problem: models invent specifics |
| 4 | **RAG in ten lines** — the whole idea, on one screen |
| 5 | The same thing properly: documents, chunks, embeddings, search, prompt |
| 6 | Without RAG vs. with RAG |
| 7 | Evaluating retrieval — before you evaluate answers |
| 8 | Is the answer actually grounded? |
| 9 | Summary |
| 10 | Exercises: everything this notebook deliberately left out |

## 1  Setup

Four packages. `numpy` and `pandas` are already in Colab.

| Package | Purpose |
|---|---|
| `transformers` | The generator pipeline |
| `sentence-transformers` | The embedder: whole texts in, one vector per text out |
| `accelerate` | Puts the weights on the GPU |
| `wikipedia-api` | A thin client for the Wikipedia API |

In [1]:
!pip install -q wikipedia-api sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 14.9 MB/s eta 0:00:00


In [2]:
from google.colab import output
output.enable_custom_widget_manager()

In [3]:
import re, warnings
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

from sentence_transformers import SentenceTransformer
from transformers import pipeline, logging as hf_logging

hf_logging.set_verbosity_error()

## 2  A Generator and an Embedder

Everything the neural side of this notebook does is covered by two models.

**The generator** writes text. We use `Qwen2.5-1.5B-Instruct`, unquantised — about 3 GB. A small model is a defensible choice for RAG in a way it would not be for open-ended writing: here the model's job is to *read a passage it has been handed*, not to recall facts from its weights.

**The embedder** turns text into a vector. `all-MiniLM-L6-v2` is a 22 M-parameter transformer producing 384 numbers per text. We load it with the `sentence-transformers` library, whose `encode()` takes whole texts and returns one vector for each.

> On an older GPU such as a Colab T4, `bfloat16` is emulated and slow — pass `dtype="float16"` instead of `"auto"`.

In [4]:
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                dtype="auto", device_map="auto")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # uses the GPU if there is one


def ask_llm(system, user, max_new_tokens=400, **kwargs):
    """Send a chat to the generator and return the reply."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    out = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False, **kwargs)
    return out[0]["generated_text"][-1]["content"]


def embed(texts):
    """Embed texts into a matrix of unit-length vectors, one row each."""
    return embedder.encode(texts, normalize_embeddings=True)


print(f"generator on {pipe.model.device}, embedder gives {embed(['hello']).shape[1]} dimensions")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

generator on cuda:0, embedder gives 384 dimensions


In [5]:
print(ask_llm("You are a helpful assistant for questions about Lipizzaner horses. Answer concisely.",
              "In what year was the founding stallion Siglavy foaled, and where did he come from?"))

Siglavy, the founding stallion of the Lipizzaner breed, was foaled in 1702 at the court of Emperor Charles VI in Vienna, Austria. He came from the family of Count Franz von Habsburg-Lorraine.


Fluent, specific, and you have no way to tell whether it is true.

## 4  RAG in Ten Lines

Before any detail, here is the entire idea on a corpus of three sentences. Everything in section 5 is this, with more documents and better plumbing.

In [6]:
CORPUS = [
    "The Lipizzan breed descends from six foundation stallions. Siglavy, a grey Arabian, "
    "was foaled in 1810 in Syria and founded the Siglavy line.",
    "Idrija in western Slovenia held one of the largest mercury mines in the world, "
    "worked from 1490 until its closure in 1995.",
    "Primož Trubar published the first printed books in Slovene, the Catechismus and "
    "the Abecedarium, in 1550.",
]

question = "In what year was the founding stallion Siglavy foaled, and where did he come from?"

vectors = embed(CORPUS)                       # 1. index the corpus (once)
scores = vectors @ embed([question])[0]       # 2. compare the question against every document
context = CORPUS[int(np.argmax(scores))]      # 3. keep what is closest

print(scores)
print(f"Answer using only this context:\n\n{context}\n\n{question}\n\n")

print("LLM answer:")
print(ask_llm(f"Answer using only this context:\n\n{context}", question))   # 4. answer from it

[0.7357149  0.01388304 0.14118895]
Answer using only this context:

The Lipizzan breed descends from six foundation stallions. Siglavy, a grey Arabian, was foaled in 1810 in Syria and founded the Siglavy line.

In what year was the founding stallion Siglavy foaled, and where did he come from?


LLM answer:
Siglavy, the founding stallion of the Lipizzan breed, was foaled in 1810 in Syria.


That is RAG. The rest of this notebook is about doing each of those four steps on a corpus too big to fit in a prompt — and about checking that the result is any good.

## 5  The Same Thing, Properly

### 5.1  Documents

Every RAG library has a `Document` class and every one of them is the same two fields: the text, and a dict of what you know about where it came from. The metadata is not decoration — it is what lets you filter, cite, and evaluate later.

In [7]:
@dataclass
class Document:
    page_content: str
    metadata: dict = field(default_factory=dict)

We load ten Wikipedia articles. `wikipedia-api` asks for a `user_agent` identifying you — that is Wikipedia's condition for automated access, so put a real address there.

Note `topic`: what we *asked* for, kept separately from the `title` Wikipedia *returned*, because titles redirect. Evaluation later keys off `topic`, the field we control.

In [8]:
import wikipediaapi

topics = [
    "BERT (language model)", "GPT (language model)", "Word2vec",
    "Transformer (deep learning architecture)", "Attention mechanism",
    "Transfer learning", "Natural language processing",
    "Lipizzan", "Primož Trubar", "Idrija mercury mine",
]

wiki = wikipediaapi.Wikipedia(
    user_agent="RAGWorkshop/1.0 (your-email@example.com)",  # Replace contact
    language="en",
)

all_docs = []
for topic in topics:
    page = wiki.page(topic)
    if not page.exists() or not page.text.strip():
        print(f"  skipped: {topic}")
        continue
    all_docs.append(Document(page.text[:30000],
                             {"topic": topic, "title": page.title, "source": page.fullurl}))

print(f"{len(all_docs)} articles, {sum(len(d.page_content) for d in all_docs):,} characters")

  skipped: Idrija mercury mine
9 articles, 183,504 characters


### 5.2  Chunking

An article is far too long to embed as one vector or to paste into a prompt, so we cut documents into **chunks**: passages short enough to retrieve individually.

The splitter below slides a window of `chunk_size` characters, but rather than cutting blindly it backs up to the last line break in the window's second half. Overlap repeats the tail of each chunk at the head of the next, so a fact split across a boundary survives intact somewhere.

Chunking is not neutral preprocessing: it decides what can ever be retrieved as a unit. Report it alongside the model and the prompt.

- **Too small** → chunks lack the context to answer; retrieval gets noisy.
- **Too large** → the one relevant sentence is diluted, and fewer chunks fit in the prompt.
- Start around **800 characters** (roughly 200 tokens of English).

In [9]:
def split_text(text, chunk_size=800, overlap=100):
    """Slide a window over the text, ending each chunk at a line break where possible."""
    chunks, start = [], 0
    while start < len(text):
        stop = text.rfind("\n", start + chunk_size // 2, start + chunk_size)
        stop = stop if stop > 0 else start + chunk_size
        chunks.append(text[start:stop].strip())
        start = stop - overlap
    return [c for c in chunks if c]


def split_documents(documents, **kwargs):
    """Split every document, carrying its metadata onto each chunk."""
    return [Document(piece, {**doc.metadata, "chunk": i})
            for doc in documents
            for i, piece in enumerate(split_text(doc.page_content, **kwargs))]


chunks = split_documents(all_docs, chunk_size=800, overlap=100)

print(f"{len(all_docs)} documents → {len(chunks)} chunks")
print(f"\n{chunks[0].page_content[:300]}...")

9 documents → 313 chunks

Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved t...


### 5.3  Embedding and searching

> **A vector store is a matrix of embeddings plus the list of documents its rows belong to. A search is one matrix–vector product and a sort.**

`CHUNK_VECTORS @ q` scores the question against *every* chunk at once. For a few hundred chunks that is well under a millisecond, and unlike an approximate index it never returns the wrong neighbour. Vector databases (FAISS, Chroma, pgvector) buy you sublinear search, compressed storage and concurrent writes — all of which start to matter around a million vectors, and none of which matter here.

In [10]:
CHUNK_VECTORS = embed([c.page_content for c in chunks])     # the slow step; happens once


def search(question, k=5, where=None, min_score=0.0):
    """The k chunks most similar to the question, as (chunk, score) pairs."""
    scores = CHUNK_VECTORS @ embed([question])[0]
    ranked = sorted(zip(chunks, scores), key=lambda pair: -pair[1])
    return [(chunk, float(score)) for chunk, score in ranked
            if (where is None or where(chunk.metadata)) and score >= min_score][:k]


def show(results):
    """Print retrieval results: score, source and the full chunk text."""
    if not results:
        print("(no results)")
    for chunk, score in results:
        print(f"=== {score:.3f}  {chunk.metadata['topic']}")
        print(chunk.page_content, "\n")


print(f"{CHUNK_VECTORS.shape[0]} vectors × {CHUNK_VECTORS.shape[1]} dimensions, "
      f"{CHUNK_VECTORS.nbytes / 1e6:.1f} MB")

show(search("In what year was the stallion Siglavy foaled?", k=4))

313 vectors × 384 dimensions, 0.5 MB
=== 0.660  Lipizzan
n from the Royal Danish Stud, foaled in 1765
Conversano: a black Neapolitan stallion, foaled in 1767
Maestoso: a gray stallion from the Kladrub stud with a Spanish dam, foaled 1773, descendants today all trace via Maestoso X, foaled in Hungary in 1819
Favory: a dun stallion from the Kladrub stud, foaled in 1779
Neapolitano: a bay Neapolitan stallion from the Polesine, foaled in 1790
Siglavy: a gray Arabian stallion, originally from Syria, foaled in 1810
Two additional stallion lines are found in Croatia, Hungary, and other eastern European countries, as well as in North America. They are accepted as equal to the six classical lines by the Lipizzan International Federation. These are: 

=== 0.541  Lipizzan
e accepted as equal to the six classical lines by the Lipizzan International Federation. These are:

Tulipan: A black stallion foaled about 1800 from the Croatian stud farm of Terezovac, owned by Count Janković-Bésán.
Incitato: 

### 5.4  The prompt

This is the part tutorials skip, and it is the part worth seeing: the retrieved chunks are pasted into a string, and that string is the whole of what the model receives. There is no other magic.

Two things in the instruction do most of the work — answer **only** from the context, and an explicit escape hatch for when the context does not contain the answer. Without the second, a model told to use only the context will still invent something rather than appear unhelpful.

In [11]:
RAG_SYSTEM = """You are a precise assistant. Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly: "I don't have enough information to answer that."
Do not add facts of your own. End your answer with the source number(s) you used.

CONTEXT:
{context}"""


def format_context(results):
    return "\n\n---\n\n".join(f"[Source {i}: {doc.metadata['topic']}]\n{doc.page_content}"
                              for i, (doc, _) in enumerate(results, start=1))


def rag_prompt(question, k=5):
    """Retrieve and build the system prompt. Returns (prompt, results)."""
    results = search(question, k=k)
    return RAG_SYSTEM.format(context=format_context(results)), results


def rag(question, k=5, **kwargs):
    """Retrieve, build the prompt, generate. Returns (answer, results)."""
    prompt, results = rag_prompt(question, k=k)
    return ask_llm(prompt, question, **kwargs), results

In [12]:
# Look at what the model is actually handed.
question = "In what year was the founding stallion Siglavy foaled?"
print(rag_prompt(question, k=2)[0])
print(f"\n----- and the user turn -----\n{question}")

You are a precise assistant. Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly: "I don't have enough information to answer that."
Do not add facts of your own. End your answer with the source number(s) you used.

CONTEXT:
[Source 1: Lipizzan]
n from the Royal Danish Stud, foaled in 1765
Conversano: a black Neapolitan stallion, foaled in 1767
Maestoso: a gray stallion from the Kladrub stud with a Spanish dam, foaled 1773, descendants today all trace via Maestoso X, foaled in Hungary in 1819
Favory: a dun stallion from the Kladrub stud, foaled in 1779
Neapolitano: a bay Neapolitan stallion from the Polesine, foaled in 1790
Siglavy: a gray Arabian stallion, originally from Syria, foaled in 1810
Two additional stallion lines are found in Croatia, Hungary, and other eastern European countries, as well as in North America. They are accepted as equal to the six classical lines by the Lipizzan International Federation. These are:

---

[Source

## 6  Without RAG vs. With RAG

The core experiment. Compare, in this order: does the RAG answer stay inside the retrieved text and name its source; is it more specific; and — the step everyone skips — is it *correct* when you read the chunk yourself?

In [13]:
question = "In what year was the founding stallion Siglavy foaled, and where did he come from?"
answer, results = rag(question, k=2)

print(f"WITHOUT RAG:\n{ask_llm('You are a helpful assistant. Answer concisely.', question)}\n")
print(f"WITH RAG:\n{answer}\n")
print("WHAT IT WAS SHOWN:")
show(results)

WITHOUT RAG:
Siglavy was foaled in 1972 at the stud farm of his breeder, Mr. Jozef Kowalski, located in Silesia, Poland.

WITH RAG:
The founding stallion Siglavy was foaled in 1810 in Syria.

WHAT IT WAS SHOWN:
=== 0.650  Lipizzan
n from the Royal Danish Stud, foaled in 1765
Conversano: a black Neapolitan stallion, foaled in 1767
Maestoso: a gray stallion from the Kladrub stud with a Spanish dam, foaled 1773, descendants today all trace via Maestoso X, foaled in Hungary in 1819
Favory: a dun stallion from the Kladrub stud, foaled in 1779
Neapolitano: a bay Neapolitan stallion from the Polesine, foaled in 1790
Siglavy: a gray Arabian stallion, originally from Syria, foaled in 1810
Two additional stallion lines are found in Croatia, Hungary, and other eastern European countries, as well as in North America. They are accepted as equal to the six classical lines by the Lipizzan International Federation. These are: 

=== 0.508  Lipizzan
l and François Robichon de la Guérinière, two Frenchme

## 7  Evaluating Retrieval

It is tempting to judge a RAG system by reading its answers. That conflates two different failures: the retriever handed over the wrong passage, or the retriever was fine and the model misread it. They have different fixes — and retrieval is the cheaper half to measure, because it needs no LLM at all.

The minimal evaluation is a list of questions paired with the source that *should* come back, and a **hit rate at k**. Ten hand-written questions are enough to tell a working retriever from a broken one, and writing them forces you to say what "relevant" means for your corpus.

This is the section to steal for your own work.

In [14]:
EVAL_SET = [
    ("What pre-training objective does BERT use?",                 "BERT (language model)"),
    ("How are word vectors learned in the skip-gram model?",       "Word2vec"),
    ("What problem does the attention mechanism solve?",           "Attention mechanism"),
    ("Why is a transformer faster to train than a recurrent net?", "Transformer (deep learning architecture)"),
    ("In what year was the stallion Siglavy foaled?",              "Lipizzan"),
    ("Which book did Primož Trubar publish first?",                "Primož Trubar"),
    ("What was mined at Idrija?",                                  "Idrija mercury mine"),
    ("What does fine-tuning a pre-trained model mean?",            "Transfer learning"),
]

loaded = {d.metadata["topic"] for d in chunks}
EVAL_SET = [(q, t) for q, t in EVAL_SET if t in loaded]


def evaluate(k=5):
    """Did the expected source make the top k? One row per question."""
    rows = []
    for question, expected in EVAL_SET:
        found = [doc.metadata["topic"] for doc, _ in search(question, k=k)]
        rows.append({"question": question[:52], "expected": expected[:28],
                     "hit": expected in found, "top result": found[0][:28]})
    return pd.DataFrame(rows)


results_table = evaluate(k=5)
print(f"hit@5 = {results_table['hit'].mean():.0%}")
results_table

hit@5 = 100%


,question,expected,hit,top result
0,What pre-training objective does BERT use?,BERT (language model),True,BERT (language model)
1,How are word vectors learned in the skip-gram ...,Word2vec,True,Word2vec
2,What problem does the attention mechanism solve?,Attention mechanism,True,Attention mechanism
3,Why is a transformer faster to train than a re...,Transformer (deep learning a,True,Transformer (deep learning a
4,In what year was the stallion Siglavy foaled?,Lipizzan,True,Lipizzan
5,Which book did Primož Trubar publish first?,Primož Trubar,True,Primož Trubar
6,What does fine-tuning a pre-trained model mean?,Transfer learning,True,Transformer (deep learning a


Read the misses, not only the percentage. A miss on a question naming a rare entity points at the retrieval method; a miss where the right article was found but the specific sentence was not points at the chunk size; and a miss where the answer genuinely is not in the corpus is not a miss at all — it is the corpus telling you something.

## 8  Is the Answer Grounded?

RAG reduces hallucination. It does not eliminate it. Three failure modes survive, and all three produce fluent output:

1. The right chunk was retrieved and the model misread it.
2. Nothing relevant was retrieved and the model answered from memory anyway.
3. The answer is correct but not *in* the context — right for the wrong reason, and it will not replicate.

Two cheap checks. **Abstention**: ask something the corpus cannot answer and see whether the system declines — a RAG system never observed saying "I don't know" has not been tested, only trusted. **Verification**: hand the context and the answer back and ask whether every claim is supported.

The second is "LLM-as-a-judge" and it is a weak instrument: a small model will miss subtle unsupported claims, and it cannot be the only thing between you and a published number. As a *filter* it earns its place — it flags a subset for human reading. The lexical overlap below is dumber still, but deterministic and free, and it cannot itself hallucinate.

In [15]:
JUDGE = ("You check whether an answer is supported by a context. Reply SUPPORTED if every "
         "factual claim in the answer appears in the context, or UNSUPPORTED followed by the "
         "first unsupported claim. Reply with nothing else.")

STOPWORDS = set("the a an and or of in to is are was were it its for on that this with as by".split())


def overlap(answer, results):
    """Share of the answer's content words that also occur in the retrieved context."""
    context_words = set(re.findall(r"\w+", format_context(results).lower()))
    answer_words = [w for w in re.findall(r"\w+", answer.lower())
                    if w not in STOPWORDS and len(w) > 2]
    return sum(w in context_words for w in answer_words) / max(len(answer_words), 1)


rows = []
for question in ["What pre-training objective does BERT use?",
                 "In what year was the stallion Siglavy foaled?",
                 "What is the population of Buenos Aires?",
                 "What is the capital of France?"]:
    answer, results = rag(question)
    verdict = ask_llm(JUDGE, f"CONTEXT:\n{format_context(results)}\n\n"
                             f"QUESTION: {question}\n\nANSWER: {answer}", max_new_tokens=60)
    rows.append({"question": question[:40], "answer": answer[:60] + "…",
                 "overlap": f"{overlap(answer, results):.0%}", "verdict": verdict[:40]})

pd.DataFrame(rows)

,question,answer,overlap,verdict
0,What pre-training objective does BERT us,"According to Source 4, BERT uses masked token ...",78%,SUPPORTERED
1,In what year was the stallion Siglavy fo,"According to Source 2, Siglavy was foaled in 1...",80%,SUPPORTED
2,What is the population of Buenos Aires?,I don't have enough information to answer that.…,20%,UNSUPPORTED
3,What is the capital of France?,Paris…,0%,UNSUPPORTED\nThe context does not mention


## 9  Summary

```
INDEXING (once)
  Document(text, metadata)  →  split_documents()  →  embed()  →  CHUNK_VECTORS
                                 800 chars,          MiniLM,        an (n × 384)
                                 100 overlap         normalised     NumPy matrix

QUERY (every time)
  question  →  embed()  →  CHUNK_VECTORS @ q  →  sort  →  top-k chunks
                                                              ↓
                                                    format_context()
                                                              ↓
                                              RAG_SYSTEM.format(context=…)
                                                              ↓
                                               generator  →  answer
```

**The decisions that move your results**

| Decision | Range | Effect |
|---|---|---|
| Chunk size | 200–2000 chars | Smaller = more precise; larger = more context per chunk |
| Overlap | 0–20% of chunk size | Reduces facts cut in half at a boundary |
| Embedding model | MiniLM → multilingual / E5 | Decides what can be found at all |
| `k` | 3–10 | More context helps until it dilutes |
| `min_score` | model-specific | Lets the system return nothing |
| Generator | 1.5 B → 8 B → API | Reading and synthesis quality |

**What to carry into your own work**

- Measure retrieval separately from generation. A hit rate over ten hand-written questions costs an afternoon and tells you which half is broken.
- Give the model an explicit way to decline, and check that it uses it.
- Put your own provenance metadata on every document. Filtering, citation and evaluation all depend on it.
- Chunk size, `k`, the embedding model and the prompt are part of your method, and belong in the write-up beside the model name.
- The whole retrieval half is three lines: embed the question, multiply by the matrix, sort. When you do adopt a framework, you will know which three lines it is wrapping.

## References

* HuggingFace pipelines — https://huggingface.co/docs/transformers/main_classes/pipelines
* Sentence embeddings and pooling — Reimers & Gurevych (2019) — https://arxiv.org/abs/1908.10084
* `sentence-transformers` documentation — https://sbert.net
* Original RAG paper — Lewis et al. (2020) — https://arxiv.org/abs/2005.11401
* Dense passage retrieval — Karpukhin et al. (2020) — https://arxiv.org/abs/2004.04906
* BM25 and its parameters — Robertson & Zaragoza (2009) — https://doi.org/10.1561/1500000019
* Maximal Marginal Relevance — Carbonell & Goldstein (1998) — https://doi.org/10.1145/290941.291025
* Reciprocal Rank Fusion — Cormack et al. (2009) — https://doi.org/10.1145/1571941.1572114
* Lost in the middle — Liu et al. (2023) — https://arxiv.org/abs/2307.03172
* MTEB embedding-model leaderboard — https://huggingface.co/spaces/mteb/leaderboard
* `wikipedia-api` — https://pypi.org/project/Wikipedia-API/